# 层和块

In [1]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

X = torch.rand(2, 20)
net(X)

tensor([[-0.2485, -0.0037,  0.0734,  0.1419,  0.0735,  0.1093, -0.0298, -0.0383,
         -0.0488, -0.1066],
        [-0.0718, -0.1167,  0.1665,  0.0918, -0.0186,  0.2055, -0.0554, -0.0446,
         -0.0756, -0.0261]], grad_fn=<AddmmBackward0>)

In [2]:
# 自定义块
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        return self.out(F.relu(self.hidden(X)))

In [3]:
net = MLP()
net(X)

tensor([[ 0.0463,  0.0972, -0.0867,  0.1387,  0.2591,  0.2553,  0.2369,  0.1636,
          0.4553, -0.0207],
        [-0.0295,  0.2881, -0.2282,  0.2045,  0.0885,  0.1980,  0.2928,  0.1455,
          0.4224,  0.0060]], grad_fn=<AddmmBackward0>)

In [4]:
# 顺序块
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for block in args:
            self._modules[block] = block
    def forward(self, X):
        for block in self._modules.values():
            X = block(X)
        return X

net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net(X)

tensor([[ 0.0606,  0.2472,  0.2568, -0.0434, -0.1543, -0.0778, -0.3406,  0.0202,
         -0.1246, -0.1216],
        [ 0.1156,  0.2030,  0.2124, -0.1004, -0.0552, -0.0127, -0.3853,  0.1625,
         -0.0325, -0.2040]], grad_fn=<AddmmBackward0>)

In [5]:
# 在正向传播函数中执行代码
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        X = F.relu(torch.mm(X, self.rand_weight) + 1)
        X = self.linear(X)
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

net = FixedHiddenMLP()
net(X)

tensor(0.2086, grad_fn=<SumBackward0>)

In [7]:
# 混合搭配各种组合块的方法
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, X):
        return self.linear(self.net(X))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(X)

tensor(-0.2412, grad_fn=<SumBackward0>)